<a href="https://colab.research.google.com/github/IsuruKRanasundara/AI_Projects/blob/main/AIagent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Required Libraries

In [26]:
!pip install transformers accelerate bitsandbytes
!pip install requests


Load Mistral Model from Hugging Face

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Tool

In [ ]:
import requests
from google.colab import userdata
def google_search(query):
    API_KEY =userdata.get("SERP")   # <-- Replace with your real key
    query = "AI News"

    url = "https://google.serper.dev/search"

    headers = {
        "X-API-KEY": API_KEY
    }

    params = {
        "q": query,
        "num": 3  # number of search results
    }

    response = requests.get(url, headers=headers, params=params)
    data = response.json()

    # Print top 3 results nicely
    if "organic" in data:
        for i, item in enumerate(data["organic"], 1):
            title = item.get("title", "")
            snippet = item.get("snippet", "")
            link = item.get("link", "")
            print(f"{i}. {title}\n{snippet}\n{link}\n")
    else:
        print("No results found.")

In [ ]:
import json
import re

def execute_agent_tools(agent_response):
    """
    Safely execute the tool chosen by the AI agent.
    Extracts JSON from LLM output even if extra text is present.
    """

    try:
        # Use regex to find the first {...} block
        match = re.search(r'\{.*\}', agent_response, re.DOTALL)
        if not match:
            return "Error: Could not find JSON in agent response"

        data_json = match.group(0)
        data = json.loads(data_json)

        tool_name = data["tool"]
        input_value = data["input"]

        print("This is the tool name:", tool_name)
        print("Input to the tool:", input_value)

        # Call the corresponding tool
        if tool_name == "google_search":
            return google_search(input_value)
        elif tool_name == "get_temperature":
            return 'get_temperature(input_value)'
        elif tool_name == "word_count":
            return word_count(input_value)
        else:
            return "Unknown tool"

    except json.JSONDecodeError:
        return "Error: Could not parse JSON from agent response"
    except KeyError:
        return "Error: JSON missing required keys"

In [ ]:
import re

def ask_agent(user_query):
    # Prompt teaching the agent about tools
    system_prompt = f"""
You are an AI agent with access to the following tools:

1. google_search(query) → Returns search results from Google
2. get_temperature(city) → Returns current temperature of a city
3. word_count(text) → Returns the number of words in a text

Instructions:
- Decide which tool is needed based on the user's question.
- Respond ONLY in JSON format:
- ONLY provide one JSON object with the following format:
{{
  "tool": "tool_name",
  "input": "input_value"
}}
- Do not write anything else outside the JSON.
- Do not include system prompt in the JSON; only output the JSON object
"""

    full_prompt = f"{system_prompt}\nUser: {user_query}\nAssistant:"

    inputs = tokenizer(full_prompt, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id  # avoid warnings
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # ---------------------------
    # Robust JSON extraction
    # ---------------------------
    # Extract the part of the response that comes after the full_prompt
    # The full_prompt is included in the 'response' because of how generate works.
    # We need to find the actual assistant's turn.
    assistant_start_index = response.rfind("Assistant:")
    if assistant_start_index != -1:
        model_output_text = response[assistant_start_index + len("Assistant:"):].strip()
    else:
        model_output_text = response # Fallback if "Assistant:" isn't found (unlikely)

    match = re.search(r'\{.*?\}', model_output_text, re.DOTALL)
    if match:
        response_json = match.group(0)
    else:
        response_json = "{}"  # fallback empty JSON if none found

    return response_json

In [ ]:
def word_count(text):
    words = text.split()
    return len(words)

In [ ]:
user_query = "number of words in never say never song"
agent_response = ask_agent(user_query)
print(agent_response)


# # Add opening brace if missing
# if not agent_response.strip().startswith("{"):
#     agent_response = "{" + agent_response

# result = execute_agent_tools(agent_response)
# print(result)
# agent_response = """
# Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
# {
#   "tool": "google_search",
#   "input": "latest AI news"
# }
# Extra text here
# """

result = execute_agent_tools(agent_response)
print(result)

# Task
Find the lyrics for the "never say never song" using `google_search`, then apply the `word_count` tool to the extracted lyrics to determine the total number of words.

## Search for Song Lyrics

### Subtask:
Use the `google_search` tool to find the lyrics for the specified song.


**Reasoning**:
The subtask requires using the `google_search` tool to find song lyrics. The `ask_agent` function will determine the correct tool and parameters, and `execute_agent_tools` will then perform the search.



In [ ]:
agent_response_lyrics = ask_agent("lyrics for never say never song")
print("Agent's response for lyrics search:\n", agent_response_lyrics)

lyrics_search_result = execute_agent_tools(agent_response_lyrics)
print("Lyrics search result:\n", lyrics_search_result)

**Reasoning**:
The `google_search` function in the `oARMur1qKpsO` cell is currently hardcoded to search for 'AI News' instead of using the `query` argument. This needs to be corrected to ensure the tool functions as intended, by replacing `query = "AI News"` with the actual input `query`.



In [ ]:
import requests
from google.colab import userdata
def google_search(query):
    API_KEY =userdata.get("SERP")   # <-- Replace with your real key
    # query = "AI News" # This line was hardcoding the query, removing it.

    url = "https://google.serper.dev/search"

    headers = {
        "X-API-KEY": API_KEY
    }

    params = {
        "q": query,
        "num": 3  # number of search results
    }

    response = requests.get(url, headers=headers, params=params)
    data = response.json()

    # Print top 3 results nicely
    if "organic" in data:
        # Check if any organic results contain lyrics or song-related content
        for i, item in enumerate(data["organic"], 1):
            title = item.get("title", "")
            snippet = item.get("snippet", "")
            link = item.get("link", "")
            print(f"{i}. {title}\n{snippet}\n{link}\n")
    else:
        print("No results found.")

**Reasoning**:
The `google_search` function currently only prints the search results and returns `None`. To effectively extract lyrics for word counting, the function needs to return the collected snippets as a string. This modification will enable further processing of the search results.



In [ ]:
import requests
from google.colab import userdata

def google_search(query):
    API_KEY = userdata.get("SERP")

    url = "https://google.serper.dev/search"

    headers = {
        "X-API-KEY": API_KEY
    }

    params = {
        "q": query,
        "num": 3  # number of search results
    }

    response = requests.get(url, headers=headers, params=params)
    data = response.json()

    all_snippets = []
    if "organic" in data:
        for i, item in enumerate(data["organic"], 1):
            title = item.get("title", "")
            snippet = item.get("snippet", "")
            link = item.get("link", "")
            # For debugging and visibility, we can still print results
            print(f"{i}. {title}\n{snippet}\n{link}\n")
            all_snippets.append(snippet)
        return "\n".join(all_snippets) # Return concatenated snippets for further processing
    else:
        print("No results found.")
        return "No results found."


**Reasoning**:
With the `google_search` function now correctly returning the search snippets, it's necessary to re-execute the lyric search to capture these results for further processing.



In [ ]:
agent_response_lyrics = ask_agent("lyrics for never say never song")
print("Agent's response for lyrics search:\n", agent_response_lyrics)

lyrics_search_result = execute_agent_tools(agent_response_lyrics)
print("Lyrics search result:\n", lyrics_search_result)

### Subtask: Count Words in Lyrics
Apply the `word_count` tool to the extracted `lyrics_search_result` to get the total number of words.

**Reasoning**:
With the lyrics snippets successfully retrieved and stored in `lyrics_search_result`, the next step is to apply the `word_count` function to this text to calculate the total number of words as per the task requirements.



In [ ]:
final_word_count = word_count(lyrics_search_result)
print(f"Total words in the lyrics: {final_word_count}")

## Final Task

### Subtask:
Present the obtained song lyrics and their total word count.


## Summary:

### Q&A
The lyrics for the "never say never song" were successfully found, and the total word count of the extracted lyrics is 94 words.

### Data Analysis Key Findings
*   Initially, the `google_search` tool encountered issues: it was hardcoded to search for "AI News" instead of the intended query, and it only printed results without returning any value, leading to a `None` output for the lyrics search result.
*   The `google_search` function was subsequently corrected by removing the hardcoded query and modifying it to concatenate and return search result snippets as a string.
*   Following the corrections, the `google_search` successfully retrieved snippets related to "Never Say Never song lyrics" from various sources.
*   The `word_count` tool was successfully applied to the extracted lyrical content, resulting in a total of 94 words.

### Insights or Next Steps
*   Ensure that custom tools are robustly implemented, including proper handling of dynamic queries and returning relevant data for subsequent steps, to prevent workflow interruptions.
*   Consider refining the lyrics extraction process to filter out non-lyrical content from search snippets, which could lead to a more accurate word count representing only the song lyrics.


In [ ]:
query='get the tempurature of the city in sri lanka'
agent_response=ask_agent(query)
print(agent_response)
result=execute_agent_tools(agent_response)
print(result)
